%md
# DataPulse — Silver Transform (schema-aligned workshop)

This reusable notebook covers all 25 DataPulse datasets using the column names and suggested types documented in the saved DataPulse reference PDFs for Google Ads, Amazon Ads, Magento, and SAP.

**Meta Ads note:** The five Meta dataset names were defined in the project catalogue, but individual Meta column-dictionary PDFs were not found in the saved reference set. Meta rules below are explicitly provisional and must be verified against the actual extract.

These are proposed learning schemas, not verified vendor payload contracts. Source files, selected API fields, versions, and logical grains must be checked before production.


In [0]:
dbutils.widgets.text("source_name", "google_ads")
dbutils.widgets.text("dataset_name", "campaigns")
dbutils.widgets.text("market", "IN")
dbutils.widgets.text("processing_date", "2026-10-28")
dbutils.widgets.text("source_table", "")
dbutils.widgets.text("target_table", "")
dbutils.widgets.text("target_catalog_schema", "main.datapulse_silver")
dbutils.widgets.text("load_type", "full") # safer for first test; incremental requires MERGE/slice-replacement
dbutils.widgets.text("run_id", "")

source_name = dbutils.widgets.get("source_name").strip().lower()
dataset_name = dbutils.widgets.get("dataset_name").strip().lower()
market = dbutils.widgets.get("market").strip().upper()
processing_date = dbutils.widgets.get("processing_date").strip()
source_table = dbutils.widgets.get("source_table").strip()
target_table = dbutils.widgets.get("target_table").strip()
target_catalog_schema = dbutils.widgets.get("target_catalog_schema").strip()
load_type = dbutils.widgets.get("load_type").strip().lower()
run_id = dbutils.widgets.get("run_id").strip()


%md
## 1. Schema and transformation registry

`types` maps exact documented column names to their Silver target types. `required` and `keys` are candidate rules; verify them against the actual landed extract before enforcing them.


In [0]:
RULES = {
    # GOOGLE ADS — names/types from DataPulse reference PDFs
    ("google_ads","campaigns"): {
        "required":["customer_id","campaign_id"],
        "keys":["customer_id","campaign_id"],
        "types":{"customer_id":"STRING","campaign_id":"STRING","campaign_resource_name":"STRING","campaign_name":"STRING","campaign_status":"STRING","advertising_channel_type":"STRING","advertising_channel_sub_type":"STRING","campaign_budget_resource_name":"STRING","bidding_strategy_type":"STRING","start_date":"DATE","end_date":"DATE","campaign_primary_status":"STRING","serving_status":"STRING","target_country":"STRING","business_brand":"STRING","product_category":"STRING","business_objective":"STRING","source_system":"STRING","source_extract_timestamp":"TIMESTAMP","ingestion_timestamp":"TIMESTAMP","source_file_name":"STRING","run_id":"STRING","batch_id":"STRING"},
        "dates":["start_date","end_date"], "timestamps":["source_extract_timestamp","ingestion_timestamp"], "nonnegative":[],
    },
    ("google_ads","ad_groups"): {
        "required":["customer_id","campaign_id","ad_group_id"],
        "keys":["customer_id","ad_group_id"],
        "types":{"customer_id":"STRING","campaign_id":"STRING","campaign_resource_name":"STRING","ad_group_id":"STRING","ad_group_resource_name":"STRING","ad_group_name":"STRING","ad_group_status":"STRING","ad_group_type":"STRING","cpc_bid_micros":"LONG","cpm_bid_micros":"LONG","target_cpa_micros":"LONG","business_brand":"STRING","product_category":"STRING","source_system":"STRING","source_extract_timestamp":"TIMESTAMP","ingestion_timestamp":"TIMESTAMP","source_file_name":"STRING","run_id":"STRING","batch_id":"STRING"},
        "dates":[], "timestamps":["source_extract_timestamp","ingestion_timestamp"], "nonnegative":["cpc_bid_micros","cpm_bid_micros","target_cpa_micros"],
    },
    ("google_ads","ads"): {
        "required":["customer_id","ad_group_id","ad_id"],
        "keys":["customer_id","ad_group_id","ad_id"],
        "types":{"customer_id":"STRING","campaign_id":"STRING","ad_group_id":"STRING","ad_id":"STRING","ad_resource_name":"STRING","ad_group_ad_status":"STRING","ad_type":"STRING","ad_final_url":"STRING","ad_strength":"STRING","headline_assets":"PRESERVE","description_assets":"PRESERVE","business_brand":"STRING","source_system":"STRING","source_extract_timestamp":"TIMESTAMP","ingestion_timestamp":"TIMESTAMP","source_file_name":"STRING","run_id":"STRING","batch_id":"STRING"},
        "dates":[], "timestamps":["source_extract_timestamp","ingestion_timestamp"], "nonnegative":[],
    },
    ("google_ads","keywords"): {
        "required":["customer_id","ad_group_id","ad_group_criterion_id"],
        "keys":["customer_id","ad_group_id","ad_group_criterion_id"],
        "types":{"customer_id":"STRING","campaign_id":"STRING","ad_group_id":"STRING","ad_group_criterion_id":"STRING","keyword_text":"STRING","match_type":"STRING","criterion_status":"STRING","is_negative":"BOOLEAN","cpc_bid_micros":"LONG","quality_score":"INT","business_brand":"STRING","product_category":"STRING","source_system":"STRING","source_extract_timestamp":"TIMESTAMP","ingestion_timestamp":"TIMESTAMP","source_file_name":"STRING","run_id":"STRING","batch_id":"STRING","keyword_resource_name":"STRING","criterion_type":"STRING"},
        "dates":[], "timestamps":["source_extract_timestamp","ingestion_timestamp"], "nonnegative":["cpc_bid_micros","quality_score"],
    },
    ("google_ads","campaign_daily_performance"): {
        "required":["customer_id","campaign_id","report_date"],
        "keys":["customer_id","campaign_id","report_date"],
        "types":{"customer_id":"STRING","campaign_id":"STRING","report_date":"DATE","campaign_name":"STRING","impressions":"LONG","clicks":"LONG","cost_micros":"LONG","conversions":"DOUBLE","conversions_value":"DOUBLE","source_system":"STRING","source_extract_timestamp":"TIMESTAMP","ingestion_timestamp":"TIMESTAMP","source_file_name":"STRING","run_id":"STRING","batch_id":"STRING"},
        "dates":["report_date"], "timestamps":["source_extract_timestamp","ingestion_timestamp"], "nonnegative":["impressions","clicks","cost_micros","conversions","conversions_value"],
    },

    # AMAZON ADS — names/types from DataPulse reference PDFs
    ("amazon_ads","campaigns"): {
        "required":["profile_id","campaign_id"], "keys":["profile_id","campaign_id"],
        "types":{"profile_id":"STRING","account_id":"STRING","campaign_id":"STRING","campaign_name":"STRING","campaign_type":"STRING","targeting_type":"STRING","campaign_status":"STRING","daily_budget":"DECIMAL(18,2)","currency_code":"STRING","start_date":"DATE","end_date":"DATE","portfolio_id":"STRING","marketplace":"STRING","source_system":"STRING","source_extract_timestamp":"TIMESTAMP","ingestion_timestamp":"TIMESTAMP","source_file_name":"STRING","run_id":"STRING","batch_id":"STRING"},
        "dates":["start_date","end_date"], "timestamps":["source_extract_timestamp","ingestion_timestamp"], "nonnegative":["daily_budget"],
    },
    ("amazon_ads","ad_groups"): {
        "required":["profile_id","campaign_id","ad_group_id"], "keys":["profile_id","campaign_id","ad_group_id"],
        "types":{"profile_id":"STRING","campaign_id":"STRING","ad_group_id":"STRING","ad_group_name":"STRING","ad_group_status":"STRING","default_bid":"DECIMAL(18,2)","currency_code":"STRING","serving_status":"STRING","creation_timestamp":"TIMESTAMP","last_updated_timestamp":"TIMESTAMP","source_system":"STRING","source_extract_timestamp":"TIMESTAMP","ingestion_timestamp":"TIMESTAMP","source_file_name":"STRING","run_id":"STRING","batch_id":"STRING"},
        "dates":[], "timestamps":["creation_timestamp","last_updated_timestamp","source_extract_timestamp","ingestion_timestamp"], "nonnegative":["default_bid"],
    },
    ("amazon_ads","product_ads"): {
        "required":["profile_id","campaign_id","ad_group_id","ad_id"], "keys":["profile_id","campaign_id","ad_group_id","ad_id"],
        "types":{"profile_id":"STRING","campaign_id":"STRING","ad_group_id":"STRING","ad_id":"STRING","advertised_asin":"STRING","advertised_sku":"STRING","ad_status":"STRING","serving_status":"STRING","creation_timestamp":"TIMESTAMP","last_updated_timestamp":"TIMESTAMP","marketplace":"STRING","source_system":"STRING","source_extract_timestamp":"TIMESTAMP","ingestion_timestamp":"TIMESTAMP","source_file_name":"STRING","run_id":"STRING","batch_id":"STRING"},
        "dates":[], "timestamps":["creation_timestamp","last_updated_timestamp","source_extract_timestamp","ingestion_timestamp"], "nonnegative":[],
    },
    ("amazon_ads","keywords"): {
        "required":["profile_id","campaign_id","ad_group_id"], "keys":["profile_id","campaign_id","ad_group_id","keyword_id"],
        "types":{"profile_id":"STRING","campaign_id":"STRING","ad_group_id":"STRING","keyword_id":"STRING","keyword_text":"STRING","match_type":"STRING","keyword_bid":"DECIMAL(18,2)","currency_code":"STRING","keyword_status":"STRING","creation_timestamp":"TIMESTAMP","last_updated_timestamp":"TIMESTAMP","source_system":"STRING","source_extract_timestamp":"TIMESTAMP","ingestion_timestamp":"TIMESTAMP","source_file_name":"STRING","run_id":"STRING","batch_id":"STRING"},
        "dates":[], "timestamps":["creation_timestamp","last_updated_timestamp","source_extract_timestamp","ingestion_timestamp"], "nonnegative":["keyword_bid"],
    },
    ("amazon_ads","campaign_daily_performance"): {
        "required":["profile_id","campaign_id","report_date"], "keys":["profile_id","campaign_id","report_date"],
        "types":{"profile_id":"STRING","campaign_id":"STRING","report_date":"DATE","impressions":"LONG","clicks":"LONG","spend":"DECIMAL(18,2)","attributed_sales":"DECIMAL(18,2)","attributed_orders":"LONG","attributed_units_sold":"LONG","currency_code":"STRING","source_system":"STRING","source_extract_timestamp":"TIMESTAMP","ingestion_timestamp":"TIMESTAMP","source_file_name":"STRING","run_id":"STRING","batch_id":"STRING"},
        "dates":["report_date"], "timestamps":["source_extract_timestamp","ingestion_timestamp"], "nonnegative":["impressions","clicks","spend","attributed_sales","attributed_orders","attributed_units_sold"],
    },

    # META ADS — PROVISIONAL, because no Meta column-dictionary PDFs were found in the saved references.
    ("meta_ads","campaigns"): {
        "provisional":True,"required":["campaign_id"],"keys":["account_id","campaign_id"],
        "types":{"account_id":"STRING","campaign_id":"STRING","campaign_name":"STRING","objective":"STRING","status":"STRING","buying_type":"STRING","daily_budget":"DECIMAL(18,2)","lifetime_budget":"DECIMAL(18,2)","start_time":"TIMESTAMP","stop_time":"TIMESTAMP","created_time":"TIMESTAMP","updated_time":"TIMESTAMP","source_system":"STRING","source_extract_timestamp":"TIMESTAMP","ingestion_timestamp":"TIMESTAMP","source_file_name":"STRING","run_id":"STRING","batch_id":"STRING"},
        "dates":[], "timestamps":["start_time","stop_time","created_time","updated_time","source_extract_timestamp","ingestion_timestamp"], "nonnegative":["daily_budget","lifetime_budget"],
    },
    ("meta_ads","ad_sets"): {
        "provisional":True,"required":["ad_set_id","campaign_id"],"keys":["account_id","campaign_id","ad_set_id"],
        "types":{"account_id":"STRING","campaign_id":"STRING","ad_set_id":"STRING","ad_set_name":"STRING","status":"STRING","optimization_goal":"STRING","billing_event":"STRING","daily_budget":"DECIMAL(18,2)","lifetime_budget":"DECIMAL(18,2)","start_time":"TIMESTAMP","end_time":"TIMESTAMP","created_time":"TIMESTAMP","updated_time":"TIMESTAMP","source_system":"STRING","source_extract_timestamp":"TIMESTAMP","ingestion_timestamp":"TIMESTAMP","source_file_name":"STRING","run_id":"STRING","batch_id":"STRING"},
        "dates":[], "timestamps":["start_time","end_time","created_time","updated_time","source_extract_timestamp","ingestion_timestamp"], "nonnegative":["daily_budget","lifetime_budget"],
    },
    ("meta_ads","ads"): {
        "provisional":True,"required":["ad_id","ad_set_id"],"keys":["account_id","ad_set_id","ad_id"],
        "types":{"account_id":"STRING","campaign_id":"STRING","ad_set_id":"STRING","ad_id":"STRING","ad_name":"STRING","status":"STRING","creative_id":"STRING","created_time":"TIMESTAMP","updated_time":"TIMESTAMP","source_system":"STRING","source_extract_timestamp":"TIMESTAMP","ingestion_timestamp":"TIMESTAMP","source_file_name":"STRING","run_id":"STRING","batch_id":"STRING"},
        "dates":[], "timestamps":["created_time","updated_time","source_extract_timestamp","ingestion_timestamp"], "nonnegative":[],
    },
    ("meta_ads","creatives"): {
        "provisional":True,"required":["creative_id"],"keys":["account_id","creative_id"],
        "types":{"account_id":"STRING","creative_id":"STRING","creative_name":"STRING","object_story_id":"STRING","page_id":"STRING","instagram_actor_id":"STRING","image_url":"STRING","video_id":"STRING","link_url":"STRING","title":"STRING","body":"STRING","created_time":"TIMESTAMP","updated_time":"TIMESTAMP","source_system":"STRING","source_extract_timestamp":"TIMESTAMP","ingestion_timestamp":"TIMESTAMP","source_file_name":"STRING","run_id":"STRING","batch_id":"STRING"},
        "dates":[], "timestamps":["created_time","updated_time","source_extract_timestamp","ingestion_timestamp"], "nonnegative":[],
    },
    ("meta_ads","daily_insights"): {
        "provisional":True,"required":["date"],"keys":["account_id","ad_id","date"],
        "types":{"account_id":"STRING","campaign_id":"STRING","ad_set_id":"STRING","ad_id":"STRING","date":"DATE","impressions":"LONG","reach":"LONG","clicks":"LONG","spend":"DECIMAL(18,2)","frequency":"DOUBLE","cpm":"DECIMAL(18,4)","cpc":"DECIMAL(18,4)","ctr":"DOUBLE","actions":"PRESERVE","action_values":"PRESERVE","source_system":"STRING","source_extract_timestamp":"TIMESTAMP","ingestion_timestamp":"TIMESTAMP","source_file_name":"STRING","run_id":"STRING","batch_id":"STRING"},
        "dates":["date"], "timestamps":["source_extract_timestamp","ingestion_timestamp"], "nonnegative":["impressions","reach","clicks","spend","frequency","cpm","cpc","ctr"],
    },

    # MAGENTO / ADOBE COMMERCE — names/types from DataPulse reference PDFs
    ("magento","customers"): {
        "required":["customer_id"],"keys":["customer_id"],
        "types":{"customer_id":"STRING","email":"STRING","first_name":"STRING","last_name":"STRING","created_at":"TIMESTAMP","updated_at":"TIMESTAMP","is_active":"BOOLEAN","store_id":"STRING","source_file_name":"STRING","ingested_at":"TIMESTAMP"},
        "dates":[], "timestamps":["created_at","updated_at","ingested_at"], "nonnegative":[],
    },
    ("magento","products"): {
        "required":["product_id"],"keys":["product_id","store_id"],
        "types":{"product_id":"STRING","sku":"STRING","product_name":"STRING","product_type":"STRING","price":"DECIMAL(18,2)","currency_code":"STRING","is_active":"BOOLEAN","created_at":"TIMESTAMP","updated_at":"TIMESTAMP","store_id":"STRING","source_file_name":"STRING","ingested_at":"TIMESTAMP"},
        "dates":[], "timestamps":["created_at","updated_at","ingested_at"], "nonnegative":["price"],
    },
    ("magento","orders"): {
        "required":["order_id"],"keys":["order_id"],
        "types":{"order_id":"STRING","customer_id":"STRING","order_number":"STRING","order_status":"STRING","order_created_at":"TIMESTAMP","currency_code":"STRING","subtotal_amount":"DECIMAL(18,2)","discount_amount":"DECIMAL(18,2)","tax_amount":"DECIMAL(18,2)","shipping_amount":"DECIMAL(18,2)","grand_total":"DECIMAL(18,2)","store_id":"STRING","source_file_name":"STRING","ingested_at":"TIMESTAMP"},
        "dates":[], "timestamps":["order_created_at","ingested_at"], "nonnegative":["subtotal_amount","tax_amount","shipping_amount","grand_total"],
    },
    ("magento","order_items"): {
        "required":["order_item_id","order_id"],"keys":["order_id","order_item_id"],
        "types":{"order_item_id":"STRING","order_id":"STRING","product_id":"STRING","sku":"STRING","product_name":"STRING","quantity_ordered":"DECIMAL(18,4)","unit_price":"DECIMAL(18,2)","discount_amount":"DECIMAL(18,2)","tax_amount":"DECIMAL(18,2)","row_total":"DECIMAL(18,2)","quantity_refunded":"DECIMAL(18,4)","source_file_name":"STRING","ingested_at":"TIMESTAMP"},
        "dates":[], "timestamps":["ingested_at"], "nonnegative":["quantity_ordered","unit_price","tax_amount","row_total","quantity_refunded"],
    },
    ("magento","order_refunds"): {
        "required":["refund_id","order_id"],"keys":["refund_id"],
        "types":{"refund_id":"STRING","order_id":"STRING","order_item_id":"STRING","refund_created_at":"TIMESTAMP","refund_status":"STRING","refund_amount":"DECIMAL(18,2)","currency_code":"STRING","refund_reason":"STRING","source_file_name":"STRING","ingested_at":"TIMESTAMP"},
        "dates":[], "timestamps":["refund_created_at","ingested_at"], "nonnegative":["refund_amount"],
    },

    # SAP logical extracts — names/types from DataPulse reference PDFs
    ("sap","customers"): {
        "required":["customer_id"],"keys":["customer_id"],
        "types":{"customer_id":"STRING","customer_name":"STRING","country":"STRING","city":"STRING","customer_group":"STRING","created_at":"TIMESTAMP","updated_at":"TIMESTAMP","source_file_name":"STRING","ingested_at":"TIMESTAMP"},
        "dates":[], "timestamps":["created_at","updated_at","ingested_at"], "nonnegative":[],
    },
    ("sap","materials"): {
        "required":["material_id"],"keys":["material_id"],
        "types":{"material_id":"STRING","material_name":"STRING","material_type":"STRING","base_unit_of_measure":"STRING","product_group":"STRING","is_active":"BOOLEAN","created_at":"TIMESTAMP","updated_at":"TIMESTAMP","source_file_name":"STRING","ingested_at":"TIMESTAMP"},
        "dates":[], "timestamps":["created_at","updated_at","ingested_at"], "nonnegative":[],
    },
    ("sap","sales_order_items"): {
        "required":["sales_order_id","item_number"],"keys":["sales_order_id","item_number"],
        "types":{"sales_order_id":"STRING","item_number":"STRING","customer_id":"STRING","material_id":"STRING","order_quantity":"DECIMAL(18,3)","unit_of_measure":"STRING","net_amount":"DECIMAL(18,2)","currency_code":"STRING","order_date":"DATE","order_status":"STRING","source_file_name":"STRING","ingested_at":"TIMESTAMP"},
        "dates":["order_date"], "timestamps":["ingested_at"], "nonnegative":["order_quantity","net_amount"],
    },
    ("sap","billing_items"): {
        "required":["billing_document_id","item_number"],"keys":["billing_document_id","item_number"],
        "types":{"billing_document_id":"STRING","item_number":"STRING","sales_order_id":"STRING","customer_id":"STRING","material_id":"STRING","billing_date":"DATE","billing_quantity":"DECIMAL(18,3)","unit_of_measure":"STRING","net_amount":"DECIMAL(18,2)","currency_code":"STRING","billing_status":"STRING","source_file_name":"STRING","ingested_at":"TIMESTAMP"},
        "dates":["billing_date"], "timestamps":["ingested_at"], "nonnegative":["billing_quantity","net_amount"],
    },
    ("sap","inventory_snapshots"): {
        "required":["material_id","plant_id","snapshot_date"],"keys":["material_id","plant_id","storage_location_id","snapshot_date"],
        "types":{"material_id":"STRING","plant_id":"STRING","storage_location_id":"STRING","stock_quantity":"DECIMAL(18,3)","unit_of_measure":"STRING","stock_type":"STRING","snapshot_date":"DATE","source_file_name":"STRING","ingested_at":"TIMESTAMP"},
        "dates":["snapshot_date"], "timestamps":["ingested_at"], "nonnegative":["stock_quantity"],
    },
}

if (source_name, dataset_name) not in RULES:
    raise ValueError(f"No schema/rules registered for {source_name}/{dataset_name}")
rules = RULES[(source_name, dataset_name)]
if load_type not in {"full","incremental"}:
    raise ValueError("load_type must be full or incremental")

source_table = source_table or f"main.datapulse_bronze.bronze_{source_name}_{dataset_name}"
target_table = target_table or f"{target_catalog_schema}.silver_{source_name}_{dataset_name}"


%md
## 2. Read Bronze and validate required columns


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.sql.window import Window
from datetime import datetime, timezone

df = spark.table(source_table)
missing = [c for c in rules["required"] if c not in df.columns]
if missing:
    raise ValueError(
        f"Required columns missing from {source_table}: {missing}. "
        f"Dataset schema may not match the reference; inspect df.printSchema() and update RULES."
    )

# Filter market only if a real market field or Bronze metadata field exists.
market_column = next((c for c in ["market", "marketplace", "target_country", "_market"] if c in df.columns), None)
if market and market_column:
    df = df.filter(F.upper(F.col(market_column).cast("string")) == market)


%md
## 3. Standardize and cast only configured columns that exist

Nested fields such as Google Ads headline/description assets and Meta insight actions are preserved rather than blindly flattened.


In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, F.trim(F.col(field.name)))

for col_name, target_type in rules["types"].items():
    if col_name not in df.columns or target_type == "PRESERVE":
        continue
    if target_type == "DATE":
        df = df.withColumn(col_name, F.to_date(F.col(col_name)))
    elif target_type == "TIMESTAMP":
        df = df.withColumn(col_name, F.to_timestamp(F.col(col_name)))
    elif target_type == "BOOLEAN":
        df = df.withColumn(col_name, F.col(col_name).cast("boolean"))
    elif target_type == "LONG":
        df = df.withColumn(col_name, F.col(col_name).cast("long"))
    elif target_type == "INT":
        df = df.withColumn(col_name, F.col(col_name).cast("int"))
    elif target_type == "DOUBLE":
        df = df.withColumn(col_name, F.col(col_name).cast("double"))
    elif target_type.startswith("DECIMAL"):
        df = df.withColumn(col_name, F.col(col_name).cast(target_type.lower()))
    elif target_type == "STRING":
        df = df.withColumn(col_name, F.col(col_name).cast("string"))

processed_at = datetime.now(timezone.utc).isoformat()
df = df.withColumn("_silver_processed_at_utc", F.lit(processed_at).cast("timestamp"))
df = df.withColumn("_silver_run_id", F.lit(run_id or None))


%md
## 4. Deduplicate using the documented candidate key

Keys are candidates from the reference documents, not guaranteed vendor primary keys. For products, history/snapshots and source updates, validate whether the key must include a version, store, marketplace, or extraction timestamp.


In [0]:
keys = [k for k in rules["keys"] if k in df.columns]
missing_key_columns = [k for k in rules["keys"] if k not in df.columns]
if missing_key_columns:
    raise ValueError(
        f"Configured key columns missing for {source_name}/{dataset_name}: {missing_key_columns}. "
        "Confirm the source schema and business grain before changing the key."
    )

ordering = next((c for c in ["last_updated_timestamp","updated_at","order_created_at","refund_created_at","ingestion_timestamp","ingested_at","_ingested_at_utc"] if c in df.columns), None)
if ordering:
    w = Window.partitionBy(*keys).orderBy(F.col(ordering).desc_nulls_last())
    df = df.withColumn("_dedupe_rn", F.row_number().over(w)).filter(F.col("_dedupe_rn") == 1).drop("_dedupe_rn")
else:
    df = df.dropDuplicates(keys)


%md
## 5. Quality metrics and quarantine null business keys


In [0]:
null_key_condition = None
for key in keys:
    cond = F.col(key).isNull()
    null_key_condition = cond if null_key_condition is None else (null_key_condition | cond)

for col_name in rules.get("nonnegative", []):
    if col_name in df.columns:
        # Report invalid negatives in a metric; do not silently drop them.
        pass

quality = df.agg(
    F.count(F.lit(1)).alias("rows_after_dedup"),
    F.sum(F.when(null_key_condition, 1).otherwise(0)).alias("null_key_rows")
).first()
rows_after_dedup = quality["rows_after_dedup"] or 0
null_key_rows = quality["null_key_rows"] or 0
df_valid = df.filter(~null_key_condition)
df_invalid = df.filter(null_key_condition)


%md
## 6. Write Silver and quarantine tables

`incremental` currently uses append only as a workshop placeholder. Before production, replace it with MERGE/upsert or a controlled replacement of the exact processing slice; append is not retry-safe.


In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_table.rsplit('.', 1)[0]}")
if load_type == "full":
    df_valid.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(target_table)
else:
    df_valid.write.format("delta").mode("append").saveAsTable(target_table)

quarantine_table = f"{target_catalog_schema}.quarantine_{source_name}_{dataset_name}"
if null_key_rows > 0:
    df_invalid = df_invalid.withColumn("_dq_reason", F.lit("NULL_BUSINESS_KEY"))
    df_invalid.write.format("delta").mode("append").saveAsTable(quarantine_table)

print({
    "status":"SUCCESS", "source_name":source_name, "dataset_name":dataset_name,
    "market":market or None, "processing_date":processing_date,
    "source_table":source_table, "target_table":target_table, "load_type":load_type,
    "candidate_keys":keys, "dedup_order_column":ordering,
    "rows_after_dedup":rows_after_dedup, "rows_written_to_silver":rows_after_dedup-null_key_rows,
    "rows_quarantined_null_keys":null_key_rows,
    "provisional_schema":bool(rules.get("provisional",False))
})


%md
## Production checklist
- Confirm source schemas and actual field names for every dataset.
- Meta Ads entries are provisional until its dataset definitions are approved.
- Confirm the candidate key/grain and latest-record rule per dataset.
- Use explicit source schemas where appropriate and test cast failures.
- Define timezone and date parsing rules; use DECIMAL for currency and document currency.
- Implement incremental MERGE/slice replacement and manifest idempotency.
- Add nonnegative/range validation and relationship checks (campaign → ad group → ad, order → order items, etc.) with quarantine/audit counts.
